Installing Phase:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Training:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm
import re

# ==== CONFIG ====
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

TRAIN_IMAGES_PER_SESSION = 4  # 01.jpg to 04.jpg
IMAGE_SIZE = (100, 300)
NUM_COMPONENTS = 137

train_data = []
train_labels = []

print("\n🚀 Preparing training data (Strategy 2 + 2DPCA)...\n")

# ==== STEP 1: LOAD TRAINING IMAGES (Strategy 2) ====
folder_list = sorted([f for f in os.listdir(base_path_sess1) if f.startswith("vein")])

for folder_name in tqdm(folder_list, desc="Processing folders"):
    match = re.match(r"vein(\d{3})_(\d)", folder_name)
    if not match:
        print(f"⚠️ Skipping unrecognized folder: {folder_name}")
        continue

    subject_id = match.group(1)
    finger_id = match.group(2)

    for session_label, base_path in [("session1", base_path_sess1), ("session2", base_path_sess2)]:
        folder_path = os.path.join(base_path, folder_name)

        for i in range(1, TRAIN_IMAGES_PER_SESSION + 1):
            img_filename = f"{i:02d}.jpg"
            img_path = os.path.join(folder_path, img_filename)

            print(f"📥 Loading image: {img_path}")  # Print the image path

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"❌ Missing image: {img_path}")
                continue

            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            class_label = f"{session_label}_subject{subject_id}_fingervein{finger_id}"
            print(f"🏷️ Assigned label: {class_label}")  # Print the label

            train_data.append(img_norm)
            train_labels.append(class_label)

# ==== STEP 2: APPLY 2DPCA ====
def compute_2dpca_projection(images_2d, num_components):
    print("\n⚙️ Computing 2DPCA projection matrix...")
    n = len(images_2d)
    h, w = images_2d[0].shape
    mean_img = sum(images_2d) / n
    G_t = np.zeros((w, w))

    for i, img in enumerate(images_2d):
        A = img - mean_img
        G_t += A.T @ A

    G_t /= n
    eig_vals, eig_vecs = np.linalg.eigh(G_t)
    idx = np.argsort(-eig_vals)
    eig_vecs = eig_vecs[:, idx[:num_components]]
    print(f"✅ 2DPCA projection matrix shape: {eig_vecs.shape}")
    return eig_vecs

W = compute_2dpca_projection(train_data, NUM_COMPONENTS)

# ==== STEP 3: PROJECT TRAIN IMAGES USING 2DPCA ====
projected_features = []
for img in train_data:
    feat = img @ W
    projected_features.append(feat)

# ==== STEP 4: FLATTEN FOR CLASSIFIER ====
flat_features = np.array([f.flatten() for f in projected_features])
train_labels = np.array(train_labels)

print("\n✅ Training data ready!")
print(f"📐 Projected feature shape: {flat_features.shape}")
print(f"🏷️ Labels shape: {train_labels.shape}")
print(f"🔎 Sample labels: {train_labels[:5]}")


Testing:

In [ ]:
import re

test_data = []
test_labels = []

print("\n🧪 Preparing test data (Strategy 2 + 2DPCA)...")

# Use images 5 and 6 as per P1 protocol (test set)
for session_label, base_path in [("session1", base_path_sess1), ("session2", base_path_sess2)]:
    folder_list = sorted([f for f in os.listdir(base_path) if f.startswith("vein")])

    for folder_name in tqdm(folder_list, desc=f"Processing {session_label} folders"):
        match = re.match(r"vein(\d{3})_(\d)", folder_name)
        if not match:
            print(f"⚠️ Skipping unrecognized folder: {folder_name}")
            continue

        subject_id = match.group(1)
        finger_id = match.group(2)

        folder_path = os.path.join(base_path, folder_name)

        for img_idx in [5, 6]:  # Test images only
            img_filename = f"{img_idx:02d}.jpg"
            img_path = os.path.join(folder_path, img_filename)

            print(f"📥 Loading test image: {img_path}")  # <-- Added print

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"❌ Missing: {img_path}")
                continue

            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            label = f"{session_label}_subject{subject_id}_fingervein{finger_id}"
            print(f"🏷️ Assigned test label: {label}")  # <-- Added print

            test_data.append(img_norm)
            test_labels.append(label)

# Convert to NumPy arrays
test_data = np.array(test_data)
test_labels = np.array(test_labels)

# ==== STEP X: PROJECT TEST IMAGES USING 2DPCA ====
proj_test_features = [img @ W for img in test_data]
flat_test_features = np.array([f.flatten() for f in proj_test_features])

print(f"\n✅ Projected test features shape: {flat_test_features.shape}")
print(f"🔍 Example test labels: {test_labels[:5]}")


Benchmarking:

In [ ]:
correct_matches = 0
total_tests = len(flat_test_features)

print("\n🔍 Starting classification using Manhattan distance...")

for i in range(total_tests):
    test_vec = flat_test_features[i]
    true_label = test_labels[i]  # e.g., "001_img5_s2"

    # Compute Manhattan distances
    distances = np.sum(np.abs(flat_features - test_vec), axis=1)
    min_index = np.argmin(distances)
    predicted_label = train_labels[min_index]  # e.g., "001_img2_s2"

    print(f"\nTest sample {i+1}:")
    print(f"  🎯 Predicted → {predicted_label}")
    print(f"  ✅ Actual    → {true_label}")

    # Extract subject ID and session from labels
    true_parts = true_label.split('_')
    pred_parts = predicted_label.split('_')

    true_id = true_parts[0]
    true_session = true_parts[-1]

    pred_id = pred_parts[0]
    pred_session = pred_parts[-1]

    if pred_id == true_id and pred_session == true_session:
        correct_matches += 1
        print("  🟢 Match (ID & Session correct)")
    else:
        print("  🔴 Mismatch")

# Final accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n🏁 Final recognition accuracy: {accuracy:.2f}%")
